# EDA — шаблон потока Insights

Ядро: `.venv` этого репо (VS Code → Select Kernel → `.venv`). Сначала `uv run python -m app.cli eda` — авто-отчёт в `outputs/eda/report.md`; здесь — то, чего в нём нет: гипотезы, сегменты, графики для питча.
Правила: данные смотрим агрегатами, не построчно; каждая находка → `docs/INSIGHTS.md` (цифра + график + действие для банка); графики → `outputs/figures/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from app.config import load_spec, resolve, DATA_RAW
from app.data import load_table, profile_frame, guess_target
from app.explain import segments, segments_markdown

pd.set_option("display.max_columns", 50, "display.width", 200)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
sorted(p.relative_to(ROOT) for p in DATA_RAW.rglob("*") if p.is_file())

In [ ]:
spec = load_spec(ROOT / "task.toml")
train = load_table(resolve(spec.train) if spec.train else sorted(DATA_RAW.glob("train*"))[0])
test_path = resolve(spec.test) if spec.test else next(iter(sorted(DATA_RAW.glob("test*"))), None)
test = load_table(test_path) if test_path else None
TARGET = spec.target or guess_target(train, test)
print(train.shape, None if test is None else test.shape, "target:", TARGET)
profile_frame(train, TARGET)

## Сегменты: где target отличается сильнее всего
`lift` — во сколько раз уровень target в сегменте выше среднего. Берём сегменты с долей ≥ 5% и lift ≥ 1.5× или ≤ 0.67×.

In [ ]:
y = train[TARGET]
if y.nunique() == 2:
    y = (y == sorted(y.unique())[-1]).astype(int)
cols = [c for c in train.columns if c != TARGET][:30]
segs = segments(train, y, cols, "binary" if y.nunique() == 2 else "regression")
from IPython.display import Markdown
Markdown(segments_markdown(segs[:6], "binary" if y.nunique() == 2 else "regression"))

## Гипотеза → проверка → график

In [ ]:
# Пример: уровень target по категории с размером сегмента. Замени колонку на свою гипотезу.
col = cols[0]
g = train.assign(_y=y).groupby(col, observed=True)["_y"].agg(["mean", "size"]).reset_index()
fig = px.bar(g.sort_values("mean"), x="mean", y=col, orientation="h", text="size",
             title=f"{TARGET} по {col}: <вывод одной фразой>")
fig.show()
# fig.write_image(FIG / "insight_1.png")  # нужен kaleido; или matplotlib: plt.savefig(FIG / "insight_1.png")